<div style="background: linear-gradient(90deg, #0b3c5d 0%, #1d6fa3 60%, #5fb3d9 100%); padding: 28px 30px; border-radius: 10px; color: white;">
  <div style="font-size: 30px; font-weight: 700; letter-spacing: 1px;">🌊 ONDAS, OLEAJE Y NIVEL DEL MAR</div>
  <div style="font-size: 19px; margin-top: 6px; opacity: 0.95;">Base de datos global de pendientes nearshore</div>
  <div style="font-size: 13px; margin-top: 14px; opacity: 0.85;">Geomatics and Ocean Engineering Group · E.T.S. Ingenieros de Caminos, Canales y Puertos · Universidad de Cantabria</div>
</div>

<p align="left">
  <img src="../images/Image_1.png" width="100">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<img src="../images/Image_2.png" width="100">
</p>

### 📑 Contenido

1. [Librerías y datos](#1.-Librerías-y-datos)
2. [Mapa de pendientes](#2.-Mapa-de-pendientes)
3. [Pendiente en un punto concreto](#3.-Pendiente-en-un-punto-concreto)

<div style="background-color: #e8f4fd; border-left: 6px solid #1d6fa3; padding: 10px 16px; border-radius: 6px;">
💡 Datos: base de datos global de pendientes <i>nearshore</i> (<a href='https://data.4tu.nl/articles/_/12712265/1'>4TU.ResearchData</a>). Sirve para consultar la pendiente <code>m</code> aproximada de una playa a partir de sus coordenadas.
</div>

---

## 1. Librerías y datos

In [1]:
import os.path as op

import numpy as np
import pandas as pd
import folium
import branca.colormap as bcm
import matplotlib
import matplotlib.pyplot as plt

DATA = op.join(op.abspath('..'), 'data')

In [2]:
global_slopes = pd.read_csv(op.join(DATA, 'nearshore_slopes.csv'))
global_slopes

,X,Y,dc,slope,error_code
0,-26.3728,-58.4052,20.485961,0.011182,6
1,-26.3890,-58.4079,20.485961,0.011312,6
2,-26.4057,-58.4098,20.485961,0.010914,6
3,-26.4227,-58.4112,20.485961,0.010500,6
4,-26.4396,-58.4123,20.485961,3.269054,0
...,...,...,...,...,...
783433,48.1141,67.6111,8.504953,NaN,4
783434,48.1203,67.6145,8.504953,0.001116,2
783435,47.8598,67.5380,8.508141,0.001609,2
783436,47.8671,67.5357,8.509773,NaN,4


Para que el mapa no pese demasiado, nos quedamos con un punto de cada 250 (y descartamos pendientes fuera de rango).

In [3]:
selected_values = global_slopes[(global_slopes['slope'] >= 0.0001) & (global_slopes['slope'] <= 0.5)]
selected_values = selected_values.iloc[::250]
print(f'{len(selected_values)} puntos seleccionados de {len(global_slopes)}')

2237 puntos seleccionados de 783438


---

## 2. Mapa de pendientes

### <span style='color:#1d6fa3'>2.1 Pendientes coloreadas</span>

In [4]:
vmin, vmax = 0, 0.04
norm = plt.Normalize(vmin=vmin, vmax=vmax)
cmap = matplotlib.colormaps['rainbow']

m = folium.Map()
for _, row in selected_values.iterrows():
    hex_color = matplotlib.colors.to_hex(cmap(norm(row['slope'])))
    folium.CircleMarker(
        location=[row['Y'], row['X']], radius=1,
        color=hex_color, fill=True, fill_color=hex_color, fill_opacity=1,
        tooltip=f"slope {row['slope']:.4f}",
    ).add_to(m)

# Leyenda
legend = bcm.LinearColormap([matplotlib.colors.to_hex(cmap(x)) for x in np.linspace(0, 1, 10)],
                            vmin=vmin, vmax=vmax, caption='Pendiente nearshore [-]')
legend.add_to(m)
m

### <span style='color:#1d6fa3'>2.2 Mapa con marcadores (información al pasar el ratón)</span>

In [5]:
m = folium.Map()
for _, row in selected_values.iterrows():
    folium.Marker(
        [row['Y'], row['X']],
        tooltip=f"slope {row['slope']:.4f} | lon {row['X']:.2f} | lat {row['Y']:.2f}",
        icon=folium.Icon(color='red'),
    ).add_to(m)
m

---

## 3. Pendiente en un punto concreto

<div style="background-color: #fff8e1; border-left: 6px solid #f7c61f; padding: 10px 16px; border-radius: 6px;">
<b>✏️ INPUTS</b> — modifica estos valores en la celda siguiente
<ul style="margin-bottom: 0;">
<li><code>X, Y</code>: coordenadas (lon, lat) de la playa</li>
</ul>
</div>

In [6]:
X, Y = -8.89, 43.33  # coordenadas de la playa de Razo

distances = np.sqrt((global_slopes['X'] - X)**2 + (global_slopes['Y'] - Y)**2)
closest_position = global_slopes.loc[distances.idxmin()]

print(f'Punto de partida:    ({X}, {Y})')
print(f'Punto más cercano:   ({closest_position.X:.3f}, {closest_position.Y:.3f})')
print(f'Pendiente en ese punto: {closest_position.slope:.4f}')

Punto de partida:    (-8.89, 43.33)
Punto más cercano:   (-8.888, 43.319)
Pendiente en ese punto: 0.0334
